# Objetivo B - Modelo Federado Para Inferencia En Banco Guatemala

Este notebook implementa el flujo del Objetivo B del proyecto PlusTI.

Se utilizarán los bancos etiquetados:
- Bolivia
- Brazil

Y se generarán inferencias para:
- Guatemala

La metodología reutiliza aprendizajes del Objetivo A, pero entrena un modelo nuevo con Bolivia + Brazil.

In [55]:
import os
import warnings

warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import (
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)

os.makedirs("resultados_objetivo_b", exist_ok=True)

print("Librerías cargadas.")
print("Carpeta resultados_objetivo_b/ lista.")

Librerías cargadas.
Carpeta resultados_objetivo_b/ lista.


## 1. Definir Rutas

In [56]:
PATHS = {
    "bolivia_limpio": "data_limpia/bolivia_limpio.csv",
    "brazil_limpio": "data_limpia/brazil_limpio.csv",
    "guatemala_limpio": "data_limpia/guatemala_limpio.csv",
    "bolivia_model": "data_limpia/bolivia_model_ready.csv",
    "brazil_model": "data_limpia/brazil_model_ready.csv",
    "guatemala_model": "data_limpia/guatemala_model_ready.csv",
}

for name, path in PATHS.items():
    print(name, "->", path, "| existe:", os.path.exists(path))

bolivia_limpio -> data_limpia/bolivia_limpio.csv | existe: True
brazil_limpio -> data_limpia/brazil_limpio.csv | existe: True
guatemala_limpio -> data_limpia/guatemala_limpio.csv | existe: True
bolivia_model -> data_limpia/bolivia_model_ready.csv | existe: True
brazil_model -> data_limpia/brazil_model_ready.csv | existe: True
guatemala_model -> data_limpia/guatemala_model_ready.csv | existe: True


## 2. Cargar Datasets

In [57]:
bolivia_limpio = pd.read_csv(PATHS["bolivia_limpio"], low_memory=False)
brazil_limpio = pd.read_csv(PATHS["brazil_limpio"], low_memory=False)
guatemala_limpio = pd.read_csv(PATHS["guatemala_limpio"], low_memory=False)

bolivia_model = pd.read_csv(PATHS["bolivia_model"], low_memory=False)
brazil_model = pd.read_csv(PATHS["brazil_model"], low_memory=False)
guatemala_model = pd.read_csv(PATHS["guatemala_model"], low_memory=False)

datasets = {
    "bolivia_limpio": bolivia_limpio,
    "brazil_limpio": brazil_limpio,
    "guatemala_limpio": guatemala_limpio,
    "bolivia_model": bolivia_model,
    "brazil_model": brazil_model,
    "guatemala_model": guatemala_model,
}

## 3. Validar Dimensiones

In [58]:
resumen_dimensiones = pd.DataFrame([
    {
        "dataset": name,
        "filas": df.shape[0],
        "columnas": df.shape[1],
    }
    for name, df in datasets.items()
])

resumen_dimensiones

,dataset,filas,columnas
0,bolivia_limpio,100003,71
1,brazil_limpio,100000,71
2,guatemala_limpio,100000,69
3,bolivia_model,100003,51
4,brazil_model,100000,52
5,guatemala_model,100000,50


## 4. Validar Targets

In [59]:
print("Bolivia target:")
print(bolivia_model["is_fraud_binary"].value_counts(dropna=False))

print("\nBrazil target:")
print(brazil_model["is_fraud_binary"].value_counts(dropna=False))

print("\nGuatemala columnas target:")
print([c for c in guatemala_model.columns if "fraud" in c.lower()])

Bolivia target:
is_fraud_binary
0    95084
1     4919
Name: count, dtype: int64

Brazil target:
is_fraud_binary
0    96795
1     3205
Name: count, dtype: int64

Guatemala columnas target:
[]


## 5. Revisar Columnas Comunes

In [60]:
cols_bolivia = set(bolivia_model.columns)
cols_brazil = set(brazil_model.columns)
cols_guatemala = set(guatemala_model.columns)

cols_comunes = sorted(cols_bolivia & cols_brazil & cols_guatemala)
cols_union = sorted(cols_bolivia | cols_brazil | cols_guatemala)

print("Columnas Bolivia:", len(cols_bolivia))
print("Columnas Brazil:", len(cols_brazil))
print("Columnas Guatemala:", len(cols_guatemala))
print("Columnas comunes:", len(cols_comunes))
print("Columnas unión:", len(cols_union))

print("\nColumnas en Brazil pero no en Bolivia:")
print(sorted(cols_brazil - cols_bolivia))

print("\nColumnas en Guatemala pero no en Bolivia:")
print(sorted(cols_guatemala - cols_bolivia))

print("\nColumnas en Bolivia pero no en Guatemala:")
print(sorted(cols_bolivia - cols_guatemala))

Columnas Bolivia: 51
Columnas Brazil: 52
Columnas Guatemala: 50
Columnas comunes: 49
Columnas unión: 52

Columnas en Brazil pero no en Bolivia:
['DE2_PAN']

Columnas en Guatemala pero no en Bolivia:
['DE2_PAN']

Columnas en Bolivia pero no en Guatemala:
['is_fraud', 'is_fraud_binary']


## 6. Definir Features Base Seguras

In [61]:
TARGET_COLS = ["is_fraud", "is_fraud_binary"]

LOCAL_OR_SENSITIVE_COLUMNS = [
    "bank_code",
    "bank_name",
    "bank_country",
    "bank_tier",
    "transaction_id",
    "client_id",
    "pan_masked",
    "pan_hash",
    "DE2_PAN",
    "DE35_track2_data_masked",
    "DE37_retrieval_reference_number",
    "DE38_authorization_code",
    "DE41_terminal_id",
    "DE42_card_acceptor_id",
    "DE102_account_id_1",
    "DE103_account_id_2",
]

feature_cols_base = sorted([
    col for col in cols_comunes
    if col not in TARGET_COLS
    and col not in LOCAL_OR_SENSITIVE_COLUMNS
])

print("Cantidad de features base:", len(feature_cols_base))
print("Features base:")
for col in feature_cols_base:
    print("-", col)

Cantidad de features base: 49
Features base:
- DE100_receiving_institution_id
- DE11_STAN
- DE123_pos_data_code
- DE12_local_time
- DE13_local_date
- DE14_expiration_date
- DE15_settlement_date
- DE18_merchant_category_code
- DE19_acquirer_country_code
- DE22_pos_entry_mode
- DE23_card_seq_number
- DE25_pos_condition_code
- DE39_response_code
- DE3_processing_code
- DE43_card_acceptor_name_location
- DE49_currency_code_transaction
- DE4_amount_transaction
- DE52_pin_data_present
- DE55_emv_data_present
- DE58_authorizing_agent_id
- DE60_pos_terminal_type
- DE61_pos_extended_data
- DE63_network_specific
- DE6_amount_cardholder_billing
- DE7_transmission_datetime
- DE9_conversion_rate_billing
- amount_diff_baseline
- amount_local
- amount_tx_currency
- amount_usd
- amount_vs_baseline
- approved
- card_brand
- channel
- client_baseline_amount
- client_home_city
- client_segment
- currency_tx_alpha
- day_of_week
- distance_from_home_km
- distance_from_home_km_was_missing
- high_amount_p95


In [62]:
X_bo_base = bolivia_model[feature_cols_base].copy()
X_br_base = brazil_model[feature_cols_base].copy()
X_gt_base = guatemala_model[feature_cols_base].copy()

print("Bolivia X base:", X_bo_base.shape)
print("Brazil X base:", X_br_base.shape)
print("Guatemala X base:", X_gt_base.shape)

assert list(X_bo_base.columns) == list(X_br_base.columns) == list(X_gt_base.columns)

print("Columnas alineadas correctamente.")

Bolivia X base: (100003, 49)
Brazil X base: (100000, 49)
Guatemala X base: (100000, 49)
Columnas alineadas correctamente.


In [63]:
resumen_features = pd.DataFrame({
    "feature": feature_cols_base,
    "dtype_bolivia": [str(X_bo_base[c].dtype) for c in feature_cols_base],
    "dtype_brazil": [str(X_br_base[c].dtype) for c in feature_cols_base],
    "dtype_guatemala": [str(X_gt_base[c].dtype) for c in feature_cols_base],
    "nulos_bolivia": [X_bo_base[c].isna().sum() for c in feature_cols_base],
    "nulos_brazil": [X_br_base[c].isna().sum() for c in feature_cols_base],
    "nulos_guatemala": [X_gt_base[c].isna().sum() for c in feature_cols_base],
    "unicos_bolivia": [X_bo_base[c].nunique(dropna=True) for c in feature_cols_base],
    "unicos_brazil": [X_br_base[c].nunique(dropna=True) for c in feature_cols_base],
    "unicos_guatemala": [X_gt_base[c].nunique(dropna=True) for c in feature_cols_base],
})

resumen_features

,feature,dtype_bolivia,dtype_brazil,dtype_guatemala,nulos_bolivia,nulos_brazil,nulos_guatemala,unicos_bolivia,unicos_brazil,unicos_guatemala
0,DE100_receiving_institution_id,float64,float64,float64,1050,1010,2983,1,1,1
1,DE11_STAN,int64,int64,int64,0,0,0,100003,100000,100000
2,DE123_pos_data_code,object,object,object,954,989,3090,8,8,8
3,DE12_local_time,int64,int64,int64,0,0,0,56346,56575,56467
4,DE13_local_date,int64,int64,int64,0,0,0,181,182,183
5,DE14_expiration_date,int64,int64,int64,0,0,0,71,70,71
6,DE15_settlement_date,float64,float64,float64,982,1025,2935,181,182,183
7,DE18_merchant_category_code,int64,int64,int64,0,0,0,26,26,26
8,DE19_acquirer_country_code,int64,int64,int64,0,0,0,8,8,8
9,DE22_pos_entry_mode,int64,int64,int64,0,0,0,7,7,7


## 7. Ingeniería Temporal Reutilizable

In [64]:
def parse_iso8583_datetime(series, year=2025):
    """
    Convierte DE7_transmission_datetime desde formato MMDDHHMMSS numérico
    hacia datetime real usando el año del dataset.
    """
    return pd.to_datetime(
        str(year) + series.astype(str).str.zfill(10),
        format="%Y%m%d%H%M%S",
        errors="coerce"
    )

In [65]:
TEMPORAL_FEATURES = [
    "time_since_last_txn_min",
    "txn_count_last_1h",
    "txn_count_last_24h",
    "amount_zscore_customer",
]


def add_temporal_features(df_limpio, train_months=(1, 2, 3, 4, 5), year=2025):
    """
    Crea features temporales por cliente usando el orden cronológico de transacciones.

    Requiere columnas:
    - DE7_transmission_datetime
    - client_id
    - amount_usd

    Retorna un DataFrame con:
    - txn_dt
    - time_since_last_txn_min
    - txn_count_last_1h
    - txn_count_last_24h
    - amount_zscore_customer
    """
    required_cols = ["DE7_transmission_datetime", "client_id", "amount_usd"]
    missing = [c for c in required_cols if c not in df_limpio.columns]
    if missing:
        raise ValueError(f"Faltan columnas requeridas: {missing}")

    df_temp = df_limpio[required_cols].copy()

    df_temp["txn_dt"] = parse_iso8583_datetime(
        df_temp["DE7_transmission_datetime"],
        year=year
    )

    if df_temp["txn_dt"].isna().any():
        n_bad = df_temp["txn_dt"].isna().sum()
        raise ValueError(f"No se pudieron parsear {n_bad} timestamps.")

    df_temp["amount_usd"] = pd.to_numeric(df_temp["amount_usd"], errors="coerce")
    df_temp["amount_usd"] = df_temp["amount_usd"].fillna(df_temp["amount_usd"].median())

    sorted_df = df_temp.sort_values(["client_id", "txn_dt"]).copy()

    sorted_df["time_since_last_txn_min"] = (
        sorted_df.groupby("client_id")["txn_dt"]
        .diff()
        .dt.total_seconds()
        .div(60)
    )

    median_time = sorted_df["time_since_last_txn_min"].median()
    sorted_df["time_since_last_txn_min"] = (
        sorted_df["time_since_last_txn_min"].fillna(median_time)
    )

    counts_1h = []
    counts_24h = []

    for client_id, group in sorted_df.groupby("client_id", sort=False):
        g = group.set_index("txn_dt")

        count_1h = (
            g["amount_usd"]
            .rolling("60min", closed="left")
            .count()
            .fillna(0)
            .astype(int)
        )

        count_24h = (
            g["amount_usd"]
            .rolling("24H", closed="left")
            .count()
            .fillna(0)
            .astype(int)
        )

        counts_1h.append(pd.Series(count_1h.values, index=group.index))
        counts_24h.append(pd.Series(count_24h.values, index=group.index))

    sorted_df["txn_count_last_1h"] = pd.concat(counts_1h).sort_index()
    sorted_df["txn_count_last_24h"] = pd.concat(counts_24h).sort_index()

    is_train_time = sorted_df["txn_dt"].dt.month.isin(train_months)

    client_stats = (
        sorted_df.loc[is_train_time]
        .groupby("client_id")["amount_usd"]
        .agg(client_mean="mean", client_std="std")
    )

    sorted_df = sorted_df.join(client_stats, on="client_id")

    global_mean = sorted_df.loc[is_train_time, "amount_usd"].mean()
    global_std = sorted_df.loc[is_train_time, "amount_usd"].std()

    sorted_df["client_mean"] = sorted_df["client_mean"].fillna(global_mean)
    sorted_df["client_std"] = sorted_df["client_std"].fillna(global_std)
    sorted_df["client_std"] = sorted_df["client_std"].replace(0, global_std)

    sorted_df["amount_zscore_customer"] = (
        (sorted_df["amount_usd"] - sorted_df["client_mean"])
        / (sorted_df["client_std"] + 1e-10)
    )

    temporal_df = (
        sorted_df
        .sort_index()
        [["txn_dt"] + TEMPORAL_FEATURES]
        .copy()
    )

    return temporal_df

In [66]:
temporal_bo = add_temporal_features(bolivia_limpio)
temporal_br = add_temporal_features(brazil_limpio)
temporal_gt = add_temporal_features(guatemala_limpio)

print("Temporal Bolivia:", temporal_bo.shape)
print("Temporal Brazil:", temporal_br.shape)
print("Temporal Guatemala:", temporal_gt.shape)

display(temporal_bo.head())

Temporal Bolivia: (100003, 5)
Temporal Brazil: (100000, 5)
Temporal Guatemala: (100000, 5)


,txn_dt,time_since_last_txn_min,txn_count_last_1h,txn_count_last_24h,amount_zscore_customer
0,2025-01-01 00:01:51,6854.0500,0,0,0.0730
1,2025-01-01 00:03:55,6854.0500,0,0,2.7514
2,2025-01-01 00:04:10,6854.0500,0,0,-0.2469
3,2025-01-01 00:04:53,6854.0500,0,0,-0.6362
4,2025-01-01 00:07:56,6854.0500,0,0,-0.7176


In [67]:
for name, temporal_df in {
    "Bolivia": temporal_bo,
    "Brazil": temporal_br,
    "Guatemala": temporal_gt,
}.items():
    print(f"\n{name}")
    print("Rango fechas:", temporal_df["txn_dt"].min(), "->", temporal_df["txn_dt"].max())
    print("Nulos txn_dt:", temporal_df["txn_dt"].isna().sum())
    print("Distribución meses:")
    print(temporal_df["txn_dt"].dt.month.value_counts().sort_index())


Bolivia
Rango fechas: 2025-01-01 00:01:51 -> 2025-06-30 00:13:36
Nulos txn_dt: 0
Distribución meses:
txn_dt
1    17236
2    15663
3    17130
4    16615
5    17173
6    16186
Name: count, dtype: int64

Brazil
Rango fechas: 2025-01-01 00:02:21 -> 2025-07-01 01:57:51
Nulos txn_dt: 0
Distribución meses:
txn_dt
1    17160
2    15388
3    17314
4    16752
5    17318
6    16067
7        1
Name: count, dtype: int64

Guatemala
Rango fechas: 2025-01-01 00:06:28 -> 2025-07-01 15:22:18
Nulos txn_dt: 0
Distribución meses:
txn_dt
1    17345
2    15335
3    17265
4    16843
5    17131
6    16080
7        1
Name: count, dtype: int64


In [68]:
for name, temporal_df in {
    "Bolivia": temporal_bo,
    "Brazil": temporal_br,
    "Guatemala": temporal_gt,
}.items():
    print(f"\n{name}")
    print(temporal_df[TEMPORAL_FEATURES].isna().sum())
    display(temporal_df[TEMPORAL_FEATURES].describe().round(2))


Bolivia
time_since_last_txn_min    0
txn_count_last_1h          0
txn_count_last_24h         0
amount_zscore_customer     0
dtype: int64


,time_since_last_txn_min,txn_count_last_1h,txn_count_last_24h,amount_zscore_customer
count,100003.0000,100003.0000,100003.0000,100003.0000
mean,9798.4100,0.0600,0.2100,0.0200
std,9985.0300,0.4200,0.6200,1.0700
min,0.1700,0.0000,0.0000,-1.8400
25%,2772.6600,0.0000,0.0000,-0.6300
50%,6854.0500,0.0000,0.0000,-0.3900
75%,13456.4500,0.0000,0.0000,0.2100
max,126570.5200,7.0000,9.0000,39.4600



Brazil
time_since_last_txn_min    0
txn_count_last_1h          0
txn_count_last_24h         0
amount_zscore_customer     0
dtype: int64


,time_since_last_txn_min,txn_count_last_1h,txn_count_last_24h,amount_zscore_customer
count,100000.0000,100000.0000,100000.0000,100000.0000
mean,9802.0000,0.0100,0.1500,0.0300
std,9762.1000,0.0900,0.4000,1.1400
min,0.1000,0.0000,0.0000,-2.6300
25%,2893.4700,0.0000,0.0000,-0.5700
50%,6900.4200,0.0000,0.0000,-0.3100
75%,13442.8000,0.0000,0.0000,0.1700
max,114810.7200,2.0000,5.0000,30.0300



Guatemala
time_since_last_txn_min    0
txn_count_last_1h          0
txn_count_last_24h         0
amount_zscore_customer     0
dtype: int64


,time_since_last_txn_min,txn_count_last_1h,txn_count_last_24h,amount_zscore_customer
count,100000.0000,100000.0000,100000.0000,100000.0000
mean,12059.5200,0.0200,0.1500,0.0500
std,12219.6100,0.2600,0.4800,1.3500
min,0.0200,0.0000,0.0000,-2.8300
25%,3517.3800,0.0000,0.0000,-0.5800
50%,8430.9500,0.0000,0.0000,-0.2800
75%,16481.3100,0.0000,0.0000,0.2600
max,149597.7700,6.0000,7.0000,59.8800


In [69]:
X_bo_full = X_bo_base.join(temporal_bo[TEMPORAL_FEATURES])
X_br_full = X_br_base.join(temporal_br[TEMPORAL_FEATURES])
X_gt_full = X_gt_base.join(temporal_gt[TEMPORAL_FEATURES])

print("X Bolivia full:", X_bo_full.shape)
print("X Brazil full:", X_br_full.shape)
print("X Guatemala full:", X_gt_full.shape)

assert list(X_bo_full.columns) == list(X_br_full.columns) == list(X_gt_full.columns)

print("Features base + temporales alineadas correctamente.")

X Bolivia full: (100003, 53)
X Brazil full: (100000, 53)
X Guatemala full: (100000, 53)
Features base + temporales alineadas correctamente.


In [70]:
feature_cols_full = X_bo_full.columns.tolist()

print("Cantidad total de features:", len(feature_cols_full))
print("Features temporales agregadas:")
print(TEMPORAL_FEATURES)

Cantidad total de features: 53
Features temporales agregadas:
['time_since_last_txn_min', 'txn_count_last_1h', 'txn_count_last_24h', 'amount_zscore_customer']


## 8. Selección Final De Features Para Modelado

Antes de entrenar modelos, se excluyen columnas que aunque están alineadas entre bancos pueden introducir ruido, fuga temporal o baja generalización.

In [71]:
MODEL_EXCLUDE_COLUMNS = [
    # Constantes o casi sin variación entre bancos
    "DE100_receiving_institution_id",
    "DE58_authorizing_agent_id",
    "DE63_network_specific",

    # Identificador de transacción ISO8583, casi único por fila
    "DE11_STAN",

    # Fechas / timestamps crudos usados para ingeniería temporal
    "DE7_transmission_datetime",
    "DE12_local_time",
    "DE13_local_date",
    "DE14_expiration_date",
    "DE15_settlement_date",

    # Montos duplicados en formato ISO8583; se conservan variables interpretables en USD/local
    "DE4_amount_transaction",
    "DE6_amount_cardholder_billing",
]

feature_cols_model = [
    col for col in feature_cols_full
    if col not in MODEL_EXCLUDE_COLUMNS
]

print("Features full:", len(feature_cols_full))
print("Columnas excluidas:", len(MODEL_EXCLUDE_COLUMNS))
print("Features modelo:", len(feature_cols_model))

print("\nColumnas excluidas presentes:")
for col in MODEL_EXCLUDE_COLUMNS:
    if col in feature_cols_full:
        print("-", col)

print("\nFeatures finales para modelado:")
for col in feature_cols_model:
    print("-", col)

Features full: 53
Columnas excluidas: 11
Features modelo: 42

Columnas excluidas presentes:
- DE100_receiving_institution_id
- DE58_authorizing_agent_id
- DE63_network_specific
- DE11_STAN
- DE7_transmission_datetime
- DE12_local_time
- DE13_local_date
- DE14_expiration_date
- DE15_settlement_date
- DE4_amount_transaction
- DE6_amount_cardholder_billing

Features finales para modelado:
- DE123_pos_data_code
- DE18_merchant_category_code
- DE19_acquirer_country_code
- DE22_pos_entry_mode
- DE23_card_seq_number
- DE25_pos_condition_code
- DE39_response_code
- DE3_processing_code
- DE43_card_acceptor_name_location
- DE49_currency_code_transaction
- DE52_pin_data_present
- DE55_emv_data_present
- DE60_pos_terminal_type
- DE61_pos_extended_data
- DE9_conversion_rate_billing
- amount_diff_baseline
- amount_local
- amount_tx_currency
- amount_usd
- amount_vs_baseline
- approved
- card_brand
- channel
- client_baseline_amount
- client_home_city
- client_segment
- currency_tx_alpha
- day_of_wee

In [72]:
X_bo_model = X_bo_full[feature_cols_model].copy()
X_br_model = X_br_full[feature_cols_model].copy()
X_gt_model = X_gt_full[feature_cols_model].copy()

y_bo = bolivia_model["is_fraud_binary"].copy()
y_br = brazil_model["is_fraud_binary"].copy()

print("X_bo_model:", X_bo_model.shape, "| y_bo:", y_bo.shape)
print("X_br_model:", X_br_model.shape, "| y_br:", y_br.shape)
print("X_gt_model:", X_gt_model.shape)

assert list(X_bo_model.columns) == list(X_br_model.columns) == list(X_gt_model.columns)
assert len(X_bo_model) == len(y_bo)
assert len(X_br_model) == len(y_br)

print("Matrices finales alineadas correctamente.")

X_bo_model: (100003, 42) | y_bo: (100003,)
X_br_model: (100000, 42) | y_br: (100000,)
X_gt_model: (100000, 42)
Matrices finales alineadas correctamente.


In [73]:
resumen_model_features = pd.DataFrame({
    "feature": feature_cols_model,
    "dtype_bolivia": [str(X_bo_model[c].dtype) for c in feature_cols_model],
    "dtype_brazil": [str(X_br_model[c].dtype) for c in feature_cols_model],
    "dtype_guatemala": [str(X_gt_model[c].dtype) for c in feature_cols_model],
    "nulos_bolivia": [X_bo_model[c].isna().sum() for c in feature_cols_model],
    "nulos_brazil": [X_br_model[c].isna().sum() for c in feature_cols_model],
    "nulos_guatemala": [X_gt_model[c].isna().sum() for c in feature_cols_model],
    "unicos_bolivia": [X_bo_model[c].nunique(dropna=True) for c in feature_cols_model],
    "unicos_brazil": [X_br_model[c].nunique(dropna=True) for c in feature_cols_model],
    "unicos_guatemala": [X_gt_model[c].nunique(dropna=True) for c in feature_cols_model],
})

resumen_model_features

,feature,dtype_bolivia,dtype_brazil,dtype_guatemala,nulos_bolivia,nulos_brazil,nulos_guatemala,unicos_bolivia,unicos_brazil,unicos_guatemala
0,DE123_pos_data_code,object,object,object,954,989,3090,8,8,8
1,DE18_merchant_category_code,int64,int64,int64,0,0,0,26,26,26
2,DE19_acquirer_country_code,int64,int64,int64,0,0,0,8,8,8
3,DE22_pos_entry_mode,int64,int64,int64,0,0,0,7,7,7
4,DE23_card_seq_number,float64,float64,float64,1038,1006,2962,3,3,3
5,DE25_pos_condition_code,int64,int64,int64,0,0,0,4,4,4
6,DE39_response_code,int64,int64,int64,0,0,0,12,12,12
7,DE3_processing_code,int64,int64,int64,0,0,0,5,5,5
8,DE43_card_acceptor_name_location,object,object,object,0,0,0,1708,1674,1599
9,DE49_currency_code_transaction,int64,int64,int64,0,0,0,6,6,6


In [74]:
constantes_model = resumen_model_features[
    (resumen_model_features["unicos_bolivia"] <= 1) |
    (resumen_model_features["unicos_brazil"] <= 1) |
    (resumen_model_features["unicos_guatemala"] <= 1)
]

constantes_model

,feature,dtype_bolivia,dtype_brazil,dtype_guatemala,nulos_bolivia,nulos_brazil,nulos_guatemala,unicos_bolivia,unicos_brazil,unicos_guatemala


## 9. Funciones Reutilizables Para Modelado Y Evaluación

Se definen funciones auxiliares para preparar datos categóricos, entrenar LightGBM y evaluar modelos bajo métricas comparables.

In [75]:
try:
    from lightgbm import LGBMClassifier
    import lightgbm as lgb

    print("LightGBM cargado correctamente.")
    print("Versión:", lgb.__version__)
except ModuleNotFoundError:
    print("LightGBM no está instalado en este entorno.")
    print("Instálalo con: pip install lightgbm")
    raise

LightGBM cargado correctamente.
Versión: 4.6.0


In [76]:
def align_categorical_columns(X_train, X_eval):
    """
    Alinea columnas categóricas entre train y evaluación.

    LightGBM puede trabajar con dtype category, pero las categorías deben
    ser consistentes entre datasets para evitar códigos distintos.
    """
    X_train_prep = X_train.copy()
    X_eval_prep = X_eval.copy()

    bool_cols = X_train_prep.select_dtypes(include="bool").columns.tolist()

    if bool_cols:
        X_train_prep[bool_cols] = X_train_prep[bool_cols].astype(int)
        X_eval_prep[bool_cols] = X_eval_prep[bool_cols].astype(int)

    cat_cols = X_train_prep.select_dtypes(include="object").columns.tolist()

    for col in cat_cols:
        X_train_prep[col] = X_train_prep[col].astype("string").fillna("DESCONOCIDO")
        X_eval_prep[col] = X_eval_prep[col].astype("string").fillna("DESCONOCIDO")

        all_categories = sorted(
            set(X_train_prep[col].unique().tolist()) |
            set(X_eval_prep[col].unique().tolist())
        )

        X_train_prep[col] = pd.Categorical(X_train_prep[col], categories=all_categories)
        X_eval_prep[col] = pd.Categorical(X_eval_prep[col], categories=all_categories)

    return X_train_prep, X_eval_prep, cat_cols

In [77]:
def evaluate_predictions(y_true, y_proba, threshold=0.5, label="modelo"):
    """
    Evalúa predicciones probabilísticas con un threshold dado.
    """
    y_pred = (np.asarray(y_proba) >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    precision = tp / (tp + fp + 1e-10)
    recall = tp / (tp + fn + 1e-10)
    f1 = 2 * precision * recall / (precision + recall + 1e-10)
    ratio_fp = fp / (tp + fp + 1e-10)

    try:
        auc = roc_auc_score(y_true, y_proba)
    except ValueError:
        auc = np.nan

    return {
        "modelo": label,
        "threshold": round(float(threshold), 4),
        "auc_roc": round(float(auc), 4) if not np.isnan(auc) else np.nan,
        "precision_fraude": round(float(precision), 4),
        "recall_fraude": round(float(recall), 4),
        "f1_fraude": round(float(f1), 4),
        "ratio_falsos_positivos": round(float(ratio_fp), 4),
        "TP": int(tp),
        "FP": int(fp),
        "FN": int(fn),
        "TN": int(tn),
        "alertas_totales": int(tp + fp),
    }

In [78]:
def sweep_thresholds(y_true, y_proba, thresholds=None, label="modelo"):
    """
    Evalúa un conjunto de thresholds para analizar el trade-off entre
    recall y falsos positivos.
    """
    if thresholds is None:
        thresholds = np.concatenate([
            np.linspace(0.001, 0.009, 9),
            np.linspace(0.01, 0.99, 99),
        ])

    rows = [
        evaluate_predictions(y_true, y_proba, threshold=t, label=label)
        for t in thresholds
    ]

    return pd.DataFrame(rows)


def find_threshold_by_recall(y_true, y_proba, target_recall=0.90, label="modelo"):
    """
    Selecciona el threshold cuyo recall está más cerca del objetivo.
    En caso de empate, elige el menor ratio de falsos positivos.
    """
    df_th = sweep_thresholds(y_true, y_proba, label=label)
    df_th["dist_recall"] = (df_th["recall_fraude"] - target_recall).abs()

    best = (
        df_th
        .sort_values(["dist_recall", "ratio_falsos_positivos"], ascending=[True, True])
        .iloc[0]
    )

    return float(best["threshold"]), df_th

In [79]:
def train_lgbm_model(X_train, y_train, X_eval=None, y_eval=None, random_state=42):
    """
    Entrena LightGBM con configuración base inspirada en Objetivo A.
    """
    neg = (y_train == 0).sum()
    pos = (y_train == 1).sum()
    scale_pos_weight = neg / max(pos, 1)

    if X_eval is not None:
        X_train_prep, X_eval_prep, cat_cols = align_categorical_columns(X_train, X_eval)
    else:
        X_train_prep = X_train.copy()
        X_eval_prep = None

        bool_cols = X_train_prep.select_dtypes(include="bool").columns.tolist()
        if bool_cols:
            X_train_prep[bool_cols] = X_train_prep[bool_cols].astype(int)

        cat_cols = X_train_prep.select_dtypes(include="object").columns.tolist()
        for col in cat_cols:
            X_train_prep[col] = X_train_prep[col].astype("string").fillna("DESCONOCIDO")
            X_train_prep[col] = X_train_prep[col].astype("category")

    model = LGBMClassifier(
        objective="binary",
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_samples=20,
        scale_pos_weight=scale_pos_weight,
        random_state=random_state,
        verbosity=-1,
    )

    model.fit(
        X_train_prep,
        y_train,
        categorical_feature=cat_cols if cat_cols else "auto",
    )

    metadata = {
        "scale_pos_weight": scale_pos_weight,
        "cat_cols": cat_cols,
        "features": X_train_prep.columns.tolist(),
    }

    return model, X_train_prep, X_eval_prep, metadata

In [80]:
def run_training_scenario(
    scenario_name,
    X_train,
    y_train,
    X_eval,
    y_eval,
    target_recall=0.90,
    random_state=42
):
    """
    Entrena un modelo y evalúa:
    - threshold 0.5
    - threshold ajustado a recall objetivo
    """
    model, X_train_prep, X_eval_prep, metadata = train_lgbm_model(
        X_train=X_train,
        y_train=y_train,
        X_eval=X_eval,
        y_eval=y_eval,
        random_state=random_state,
    )

    y_proba = model.predict_proba(X_eval_prep)[:, 1]

    threshold_target, df_thresholds = find_threshold_by_recall(
        y_eval,
        y_proba,
        target_recall=target_recall,
        label=scenario_name,
    )

    metrics_default = evaluate_predictions(
        y_eval,
        y_proba,
        threshold=0.5,
        label=f"{scenario_name} | threshold=0.5",
    )

    metrics_target = evaluate_predictions(
        y_eval,
        y_proba,
        threshold=threshold_target,
        label=f"{scenario_name} | threshold~recall {target_recall}",
    )

    result = {
        "scenario_name": scenario_name,
        "model": model,
        "X_train_prep": X_train_prep,
        "X_eval_prep": X_eval_prep,
        "y_proba": y_proba,
        "threshold_target": threshold_target,
        "df_thresholds": df_thresholds,
        "metadata": metadata,
        "metrics": [metrics_default, metrics_target],
    }

    return result

In [81]:
print("Funciones listas para modelado:")
print("- align_categorical_columns")
print("- evaluate_predictions")
print("- sweep_thresholds")
print("- find_threshold_by_recall")
print("- train_lgbm_model")
print("- run_training_scenario")

Funciones listas para modelado:
- align_categorical_columns
- evaluate_predictions
- sweep_thresholds
- find_threshold_by_recall
- train_lgbm_model
- run_training_scenario


## 10. Validación Temporal Interna Por Banco

Se evalúa cada banco etiquetado entrenando con enero-mayo 2025 y testeando en junio 2025.
Esto simula el uso real del modelo: aprender del pasado para detectar fraude futuro.

In [82]:
bo_train_mask = temporal_bo["txn_dt"].dt.month.isin([1, 2, 3, 4, 5])
bo_test_mask = temporal_bo["txn_dt"].dt.month == 6

br_train_mask = temporal_br["txn_dt"].dt.month.isin([1, 2, 3, 4, 5])
br_test_mask = temporal_br["txn_dt"].dt.month == 6

print("Bolivia train:", bo_train_mask.sum())
print("Bolivia test:", bo_test_mask.sum())

print("\nBrazil train:", br_train_mask.sum())
print("Brazil test:", br_test_mask.sum())

print("\nBrazil filas fuera de train/test:", (~(br_train_mask | br_test_mask)).sum())

Bolivia train: 83817
Bolivia test: 16186

Brazil train: 83932
Brazil test: 16067

Brazil filas fuera de train/test: 1


In [83]:
X_bo_train_time = X_bo_model.loc[bo_train_mask].copy()
y_bo_train_time = y_bo.loc[bo_train_mask].copy()

X_bo_test_time = X_bo_model.loc[bo_test_mask].copy()
y_bo_test_time = y_bo.loc[bo_test_mask].copy()

X_br_train_time = X_br_model.loc[br_train_mask].copy()
y_br_train_time = y_br.loc[br_train_mask].copy()

X_br_test_time = X_br_model.loc[br_test_mask].copy()
y_br_test_time = y_br.loc[br_test_mask].copy()

print("Bolivia:")
print("X_train:", X_bo_train_time.shape, "| y_train:", y_bo_train_time.shape)
print("X_test: ", X_bo_test_time.shape, "| y_test: ", y_bo_test_time.shape)

print("\nBrazil:")
print("X_train:", X_br_train_time.shape, "| y_train:", y_br_train_time.shape)
print("X_test: ", X_br_test_time.shape, "| y_test: ", y_br_test_time.shape)

Bolivia:
X_train: (83817, 42) | y_train: (83817,)
X_test:  (16186, 42) | y_test:  (16186,)

Brazil:
X_train: (83932, 42) | y_train: (83932,)
X_test:  (16067, 42) | y_test:  (16067,)


In [84]:
def print_target_distribution(name, y_train, y_test):
    print(f"\n{name}")
    print("Train:")
    print(y_train.value_counts().sort_index())
    print(f"Fraude train: {y_train.mean() * 100:.2f}%")

    print("Test:")
    print(y_test.value_counts().sort_index())
    print(f"Fraude test: {y_test.mean() * 100:.2f}%")

print_target_distribution("Bolivia", y_bo_train_time, y_bo_test_time)
print_target_distribution("Brazil", y_br_train_time, y_br_test_time)


Bolivia
Train:
is_fraud_binary
0    79634
1     4183
Name: count, dtype: int64
Fraude train: 4.99%
Test:
is_fraud_binary
0    15450
1      736
Name: count, dtype: int64
Fraude test: 4.55%

Brazil
Train:
is_fraud_binary
0    81253
1     2679
Name: count, dtype: int64
Fraude train: 3.19%
Test:
is_fraud_binary
0    15542
1      525
Name: count, dtype: int64
Fraude test: 3.27%


In [85]:
bo_temporal_result = run_training_scenario(
    scenario_name="Bolivia temporal",
    X_train=X_bo_train_time,
    y_train=y_bo_train_time,
    X_eval=X_bo_test_time,
    y_eval=y_bo_test_time,
    target_recall=0.90,
    random_state=42,
)

pd.DataFrame(bo_temporal_result["metrics"])

,modelo,threshold,auc_roc,precision_fraude,recall_fraude,f1_fraude,ratio_falsos_positivos,TP,FP,FN,TN,alertas_totales
0,Bolivia temporal | threshold=0.5,0.5000,0.8972,0.8492,0.7649,0.8049,0.1508,563,100,173,15350,663
1,Bolivia temporal | threshold~recall 0.9,0.0100,0.8972,0.0727,0.9076,0.1347,0.9273,668,8517,68,6933,9185


In [86]:
br_temporal_result = run_training_scenario(
    scenario_name="Brazil temporal",
    X_train=X_br_train_time,
    y_train=y_br_train_time,
    X_eval=X_br_test_time,
    y_eval=y_br_test_time,
    target_recall=0.90,
    random_state=42,
)

pd.DataFrame(br_temporal_result["metrics"])

,modelo,threshold,auc_roc,precision_fraude,recall_fraude,f1_fraude,ratio_falsos_positivos,TP,FP,FN,TN,alertas_totales
0,Brazil temporal | threshold=0.5,0.5000,0.8605,0.7746,0.7333,0.7534,0.2254,385,112,140,15430,497
1,Brazil temporal | threshold~recall 0.9,0.0020,0.8605,0.0422,0.8971,0.0807,0.9578,471,10682,54,4860,11153


In [87]:
df_validacion_temporal = pd.DataFrame(
    bo_temporal_result["metrics"] +
    br_temporal_result["metrics"]
)

cols_resultados = [
    "modelo",
    "threshold",
    "auc_roc",
    "precision_fraude",
    "recall_fraude",
    "f1_fraude",
    "ratio_falsos_positivos",
    "TP",
    "FP",
    "FN",
    "TN",
    "alertas_totales",
]

df_validacion_temporal = df_validacion_temporal[cols_resultados]

df_validacion_temporal

,modelo,threshold,auc_roc,precision_fraude,recall_fraude,f1_fraude,ratio_falsos_positivos,TP,FP,FN,TN,alertas_totales
0,Bolivia temporal | threshold=0.5,0.5000,0.8972,0.8492,0.7649,0.8049,0.1508,563,100,173,15350,663
1,Bolivia temporal | threshold~recall 0.9,0.0100,0.8972,0.0727,0.9076,0.1347,0.9273,668,8517,68,6933,9185
2,Brazil temporal | threshold=0.5,0.5000,0.8605,0.7746,0.7333,0.7534,0.2254,385,112,140,15430,497
3,Brazil temporal | threshold~recall 0.9,0.0020,0.8605,0.0422,0.8971,0.0807,0.9578,471,10682,54,4860,11153


In [88]:
ruta_validacion_temporal = "resultados_objetivo_b/validacion_temporal_interna.csv"

df_validacion_temporal.to_csv(ruta_validacion_temporal, index=False)

print("Resultados guardados en:", ruta_validacion_temporal)

Resultados guardados en: resultados_objetivo_b/validacion_temporal_interna.csv


### Interpretación Validación Temporal Interna

En la validación temporal interna, Bolivia obtuvo mejor desempeño que Brazil. El AUC de Bolivia fue 0.8972, mientras que Brazil alcanzó 0.8605, lo que indica que el patrón de fraude de Bolivia es más separable bajo este conjunto de variables.

Con threshold 0.5, ambos modelos logran buena precisión, pero no alcanzan el recall objetivo cercano al 90%. Bolivia detecta 76.49% de los fraudes con ratio de falsos positivos de 0.1508, mientras que Brazil detecta 73.33% con ratio de 0.2254.

Al ajustar el threshold para acercarse a recall 0.90, el costo operativo sube mucho. Bolivia requiere threshold 0.01 y alcanza recall 0.9076, pero el ratio de falsos positivos sube a 0.9273. Brazil requiere un threshold todavía más bajo, 0.002, y alcanza recall 0.8971, pero con ratio de falsos positivos de 0.9578.

Esto muestra un trade-off fuerte: priorizar detección casi total de fraude genera demasiadas alertas falsas. Brazil parece tener peor generalización temporal que Bolivia, tanto por su menor AUC como por necesitar un threshold más agresivo para acercarse al recall objetivo.

## 11. Validación Cross-Bank

Se entrena un modelo con un banco etiquetado completo y se evalúa sobre el otro banco etiquetado completo.
Esta validación permite estimar qué tan bien generaliza el aprendizaje entre bancos con países, clientes y patrones de fraude diferentes.

In [89]:
print("Bolivia completo:")
print(y_bo.value_counts().sort_index())
print(f"Fraude Bolivia: {y_bo.mean() * 100:.2f}%")

print("\nBrazil completo:")
print(y_br.value_counts().sort_index())
print(f"Fraude Brazil: {y_br.mean() * 100:.2f}%")

Bolivia completo:
is_fraud_binary
0    95084
1     4919
Name: count, dtype: int64
Fraude Bolivia: 4.92%

Brazil completo:
is_fraud_binary
0    96795
1     3205
Name: count, dtype: int64
Fraude Brazil: 3.21%


In [90]:
bo_to_br_result = run_training_scenario(
    scenario_name="Bolivia -> Brazil",
    X_train=X_bo_model,
    y_train=y_bo,
    X_eval=X_br_model,
    y_eval=y_br,
    target_recall=0.90,
    random_state=42,
)

pd.DataFrame(bo_to_br_result["metrics"])

,modelo,threshold,auc_roc,precision_fraude,recall_fraude,f1_fraude,ratio_falsos_positivos,TP,FP,FN,TN,alertas_totales
0,Bolivia -> Brazil | threshold=0.5,0.5000,0.7779,0.8463,0.4106,0.5529,0.1537,1316,239,1889,96556,1555
1,Bolivia -> Brazil | threshold~recall 0.9,0.0100,0.7779,0.0378,0.9129,0.0726,0.9622,2926,74426,279,22369,77352


In [91]:
br_to_bo_result = run_training_scenario(
    scenario_name="Brazil -> Bolivia",
    X_train=X_br_model,
    y_train=y_br,
    X_eval=X_bo_model,
    y_eval=y_bo,
    target_recall=0.90,
    random_state=42,
)

pd.DataFrame(br_to_bo_result["metrics"])

,modelo,threshold,auc_roc,precision_fraude,recall_fraude,f1_fraude,ratio_falsos_positivos,TP,FP,FN,TN,alertas_totales
0,Brazil -> Bolivia | threshold=0.5,0.5000,0.7988,0.8489,0.3163,0.4609,0.1511,1556,277,3363,94807,1833
1,Brazil -> Bolivia | threshold~recall 0.9,0.0010,0.7988,0.0604,0.9262,0.1135,0.9396,4556,70827,363,24257,75383


In [92]:
df_validacion_cross_bank = pd.DataFrame(
    bo_to_br_result["metrics"] +
    br_to_bo_result["metrics"]
)

df_validacion_cross_bank = df_validacion_cross_bank[cols_resultados]

df_validacion_cross_bank

,modelo,threshold,auc_roc,precision_fraude,recall_fraude,f1_fraude,ratio_falsos_positivos,TP,FP,FN,TN,alertas_totales
0,Bolivia -> Brazil | threshold=0.5,0.5000,0.7779,0.8463,0.4106,0.5529,0.1537,1316,239,1889,96556,1555
1,Bolivia -> Brazil | threshold~recall 0.9,0.0100,0.7779,0.0378,0.9129,0.0726,0.9622,2926,74426,279,22369,77352
2,Brazil -> Bolivia | threshold=0.5,0.5000,0.7988,0.8489,0.3163,0.4609,0.1511,1556,277,3363,94807,1833
3,Brazil -> Bolivia | threshold~recall 0.9,0.0010,0.7988,0.0604,0.9262,0.1135,0.9396,4556,70827,363,24257,75383


In [93]:
ruta_validacion_cross_bank = "resultados_objetivo_b/validacion_cross_bank.csv"

df_validacion_cross_bank.to_csv(ruta_validacion_cross_bank, index=False)

print("Resultados guardados en:", ruta_validacion_cross_bank)

Resultados guardados en: resultados_objetivo_b/validacion_cross_bank.csv


In [94]:
df_validacion_fase3 = pd.concat(
    [
        df_validacion_temporal.assign(tipo_validacion="temporal_interna"),
        df_validacion_cross_bank.assign(tipo_validacion="cross_bank"),
    ],
    ignore_index=True
)

df_validacion_fase3 = df_validacion_fase3[
    ["tipo_validacion"] + cols_resultados
]

df_validacion_fase3

,tipo_validacion,modelo,threshold,auc_roc,precision_fraude,recall_fraude,f1_fraude,ratio_falsos_positivos,TP,FP,FN,TN,alertas_totales
0,temporal_interna,Bolivia temporal | threshold=0.5,0.5000,0.8972,0.8492,0.7649,0.8049,0.1508,563,100,173,15350,663
1,temporal_interna,Bolivia temporal | threshold~recall 0.9,0.0100,0.8972,0.0727,0.9076,0.1347,0.9273,668,8517,68,6933,9185
2,temporal_interna,Brazil temporal | threshold=0.5,0.5000,0.8605,0.7746,0.7333,0.7534,0.2254,385,112,140,15430,497
3,temporal_interna,Brazil temporal | threshold~recall 0.9,0.0020,0.8605,0.0422,0.8971,0.0807,0.9578,471,10682,54,4860,11153
4,cross_bank,Bolivia -> Brazil | threshold=0.5,0.5000,0.7779,0.8463,0.4106,0.5529,0.1537,1316,239,1889,96556,1555
5,cross_bank,Bolivia -> Brazil | threshold~recall 0.9,0.0100,0.7779,0.0378,0.9129,0.0726,0.9622,2926,74426,279,22369,77352
6,cross_bank,Brazil -> Bolivia | threshold=0.5,0.5000,0.7988,0.8489,0.3163,0.4609,0.1511,1556,277,3363,94807,1833
7,cross_bank,Brazil -> Bolivia | threshold~recall 0.9,0.0010,0.7988,0.0604,0.9262,0.1135,0.9396,4556,70827,363,24257,75383


In [95]:
ruta_validacion_fase3 = "resultados_objetivo_b/resumen_validacion_fase3.csv"

df_validacion_fase3.to_csv(ruta_validacion_fase3, index=False)

print("Resumen Fase 3 guardado en:", ruta_validacion_fase3)

Resumen Fase 3 guardado en: resultados_objetivo_b/resumen_validacion_fase3.csv


## 12. Fase 4 - Modelo 1: Centralizado Bolivia + Brazil

Se concatenan Bolivia y Brazil completos para entrenar un único modelo.
Este es el baseline federado: un solo modelo entrenado con datos de dos bancos distintos,
aplicado luego a Guatemala donde no se tienen etiquetas.

In [96]:
# Concatenar Bolivia + Brazil
X_central = pd.concat([X_bo_model, X_br_model], ignore_index=True)
y_central = pd.concat([y_bo, y_br], ignore_index=True)

print("X centralizado:", X_central.shape)
print("y centralizado:", y_central.shape)
print(f"\nFraude total: {y_central.mean() * 100:.2f}%")
print(f"  Bolivia: {y_bo.mean() * 100:.2f}%")
print(f"  Brazil:  {y_br.mean() * 100:.2f}%")

X centralizado: (200003, 42)
y centralizado: (200003,)

Fraude total: 4.06%
  Bolivia: 4.92%
  Brazil:  3.21%


### 12.1 Validación cruzada del modelo centralizado

Antes de inferir Guatemala, se valida el modelo centralizado usando cross-bank:
entrenamos con Bolivia+Brazil y evaluamos sobre cada banco por separado para
estimar qué tan bien generaliza.

In [97]:
# Validar modelo centralizado sobre Bolivia y Brazil por separado
central_vs_bo = run_training_scenario(
    scenario_name="Centralizado -> Bolivia",
    X_train=X_central,
    y_train=y_central,
    X_eval=X_bo_model,
    y_eval=y_bo,
    target_recall=0.90,
    random_state=42,
)

central_vs_br = run_training_scenario(
    scenario_name="Centralizado -> Brazil",
    X_train=X_central,
    y_train=y_central,
    X_eval=X_br_model,
    y_eval=y_br,
    target_recall=0.90,
    random_state=42,
)

df_validacion_central = pd.DataFrame(
    central_vs_bo["metrics"] +
    central_vs_br["metrics"]
)[cols_resultados]

df_validacion_central

,modelo,threshold,auc_roc,precision_fraude,recall_fraude,f1_fraude,ratio_falsos_positivos,TP,FP,FN,TN,alertas_totales
0,Centralizado -> Bolivia | threshold=0.5,0.5000,0.9990,0.8009,0.9911,0.8859,0.1991,4875,1212,44,93872,6087
1,Centralizado -> Bolivia | threshold~recall 0.9,0.7400,0.9990,0.9462,0.8971,0.9210,0.0538,4413,251,506,94833,4664
2,Centralizado -> Brazil | threshold=0.5,0.5000,0.9989,0.7927,0.9710,0.8728,0.2073,3112,814,93,95981,3926
3,Centralizado -> Brazil | threshold~recall 0.9,0.6600,0.9989,0.9232,0.9002,0.9115,0.0768,2885,240,320,96555,3125


In [98]:
# Guardar validación del modelo centralizado
df_validacion_central.to_csv(
    "resultados_objetivo_b/validacion_modelo_centralizado.csv",
    index=False
)
print("Validación centralizado guardada.")

Validación centralizado guardada.


### 12.2 Entrenar modelo centralizado final (Bolivia + Brazil completos)

Con la validación confirmada, entrenamos el modelo final sobre el 100% de Bolivia + Brazil.
Este será el modelo que se aplica a Guatemala.

In [99]:
# Entrenamiento final con Bolivia + Brazil completos (sin eval set, sin leakage)
modelo_central_final, X_central_prep, _, metadata_central = train_lgbm_model(
    X_train=X_central,
    y_train=y_central,
    X_eval=None,
    random_state=42,
)

print("Modelo centralizado entrenado.")
print(f"  scale_pos_weight: {metadata_central['scale_pos_weight']:.2f}")
print(f"  Features usadas: {len(metadata_central['features'])}")
print(f"  Columnas categóricas: {len(metadata_central['cat_cols'])}")

Modelo centralizado entrenado.
  scale_pos_weight: 23.62
  Features usadas: 42
  Columnas categóricas: 12


## 13. Fase 4 - Modelo 2: Federado Simulado

Se entrenan modelos individuales con Bolivia y Brazil por separado.
Las probabilidades sobre Guatemala se combinan con promedio ponderado,
donde el peso de cada banco se determina por su desempeño cross-bank
(el que mejor generaliza al otro banco recibe más peso).

In [100]:
# Entrenar modelos individuales (sin eval, para inferencia directa)
modelo_bo_final, X_bo_prep_final, _, metadata_bo = train_lgbm_model(
    X_train=X_bo_model,
    y_train=y_bo,
    X_eval=None,
    random_state=42,
)

modelo_br_final, X_br_prep_final, _, metadata_br = train_lgbm_model(
    X_train=X_br_model,
    y_train=y_br,
    X_eval=None,
    random_state=42,
)

print("Modelos individuales entrenados.")
print(f"  Bolivia  - scale_pos_weight: {metadata_bo['scale_pos_weight']:.2f}")
print(f"  Brazil   - scale_pos_weight: {metadata_br['scale_pos_weight']:.2f}")

Modelos individuales entrenados.
  Bolivia  - scale_pos_weight: 19.33
  Brazil   - scale_pos_weight: 30.20


In [101]:
# Pesos federados basados en AUC cross-bank (de la Fase 3)
# Bolivia -> Brazil AUC: 0.7779
# Brazil  -> Bolivia AUC: 0.7988
# El que mejor generaliza al otro recibe más peso

auc_bo_cross = 0.7779   # Bolivia entrenado, evaluado en Brazil
auc_br_cross = 0.7988   # Brazil entrenado, evaluado en Bolivia

peso_bo = auc_bo_cross / (auc_bo_cross + auc_br_cross)
peso_br = auc_br_cross / (auc_bo_cross + auc_br_cross)

print(f"Peso Bolivia:  {peso_bo:.4f}")
print(f"Peso Brazil:   {peso_br:.4f}")
print(f"Suma pesos:    {peso_bo + peso_br:.4f}")

Peso Bolivia:  0.4934
Peso Brazil:   0.5066
Suma pesos:    1.0000


## 14. Fase 5 - Preparar X_gt para inferencia

Se prepara Guatemala alineando las categorías contra el set de entrenamiento
de cada modelo antes de predecir.

In [102]:
# Preparar Guatemala para cada modelo
# Modelo centralizado
X_central_gt_prep, X_gt_central_aligned, _ = align_categorical_columns(
    X_central, X_gt_model
)

# Modelo Bolivia individual
X_bo_gt_prep, X_gt_bo_aligned, _ = align_categorical_columns(
    X_bo_model, X_gt_model
)

# Modelo Brazil individual
X_br_gt_prep, X_gt_br_aligned, _ = align_categorical_columns(
    X_br_model, X_gt_model
)

print("Guatemala preparada para los 3 modelos.")
print(f"  Shape Guatemala: {X_gt_central_aligned.shape}")

Guatemala preparada para los 3 modelos.
  Shape Guatemala: (100000, 42)


### 14.1 Generar probabilidades sobre Guatemala (100% de transacciones)

In [103]:
# Probabilidades de cada modelo sobre Guatemala completa
prob_gt_central = modelo_central_final.predict_proba(X_gt_central_aligned)[:, 1]
prob_gt_bo      = modelo_bo_final.predict_proba(X_gt_bo_aligned)[:, 1]
prob_gt_br      = modelo_br_final.predict_proba(X_gt_br_aligned)[:, 1]

# Federado simulado: promedio ponderado
prob_gt_federado = peso_bo * prob_gt_bo + peso_br * prob_gt_br

print("Probabilidades generadas.")
print(f"  prob_gt_central  - min: {prob_gt_central.min():.4f} | max: {prob_gt_central.max():.4f} | mean: {prob_gt_central.mean():.4f}")
print(f"  prob_gt_bo       - min: {prob_gt_bo.min():.4f}      | max: {prob_gt_bo.max():.4f}      | mean: {prob_gt_bo.mean():.4f}")
print(f"  prob_gt_br       - min: {prob_gt_br.min():.4f}      | max: {prob_gt_br.max():.4f}      | mean: {prob_gt_br.mean():.4f}")
print(f"  prob_gt_federado - min: {prob_gt_federado.min():.4f} | max: {prob_gt_federado.max():.4f} | mean: {prob_gt_federado.mean():.4f}")

Probabilidades generadas.
  prob_gt_central  - min: 0.0001 | max: 0.9999 | mean: 0.0570
  prob_gt_bo       - min: 0.0000      | max: 0.9999      | mean: 0.0334
  prob_gt_br       - min: 0.0000      | max: 1.0000      | mean: 0.0292
  prob_gt_federado - min: 0.0000 | max: 0.9999 | mean: 0.0312


### 14.2 Elegir threshold para Guatemala

Sin etiquetas reales en Guatemala, usamos el threshold que cada modelo
encontró en validación cross-bank para alcanzar ~90% de recall.
Elegimos el del modelo centralizado como referencia principal.

In [104]:
# Thresholds de referencia provenientes de validación cross-bank (Fase 3)
# Ajustar estos valores si la validación del modelo centralizado devuelve distintos

THRESHOLD_CENTRAL  = central_vs_bo["threshold_target"]   # viene del run_training_scenario
THRESHOLD_FEDERADO = 0.5

print(f"Threshold centralizado (recall~0.9 en cross-bank): {THRESHOLD_CENTRAL:.4f}")
print(f"Threshold federado ponderado:                       {THRESHOLD_FEDERADO:.4f}")

Threshold centralizado (recall~0.9 en cross-bank): 0.7400
Threshold federado ponderado:                       0.5000


### 14.3 Tabla comparativa de distribución de predicciones en Guatemala

In [105]:
# Comparar cuántas alertas genera cada modelo sobre Guatemala completa
def resumen_inferencia(nombre, prob, threshold):
    pred = (prob >= threshold).astype(int)
    n_fraude = pred.sum()
    n_total  = len(pred)
    return {
        "modelo": nombre,
        "threshold": round(threshold, 4),
        "predicciones_fraude": int(n_fraude),
        "predicciones_legitimas": int(n_total - n_fraude),
        "pct_fraude": round(n_fraude / n_total * 100, 2),
    }

resumen_gt = pd.DataFrame([
    resumen_inferencia("Centralizado",    prob_gt_central,  THRESHOLD_CENTRAL),
    resumen_inferencia("Federado",        prob_gt_federado, THRESHOLD_FEDERADO),
    resumen_inferencia("Bolivia solo",    prob_gt_bo,       bo_to_br_result["threshold_target"]),
    resumen_inferencia("Brazil solo",     prob_gt_br,       br_to_bo_result["threshold_target"]),
])

resumen_gt

,modelo,threshold,predicciones_fraude,predicciones_legitimas,pct_fraude
0,Centralizado,0.7400,2796,97204,2.8000
1,Federado,0.5000,2498,97502,2.5000
2,Bolivia solo,0.0100,37234,62766,37.2300
3,Brazil solo,0.0010,83463,16537,83.4600


### 14.4 Generar archivo Excel — primer 30% de Guatemala (envío a PlusTI)

El plan de trabajo exige enviar inferencias del primer 30% (30,000 filas)
en un Excel con una sola columna: is_fraud con valores True / False.
Se usa el modelo centralizado como primera entrega.

In [106]:
# Primer 30% = primeras 30,000 filas en orden original del CSV
N_30PCT = 30_000

pred_central_bool  = (prob_gt_central  >= THRESHOLD_CENTRAL).astype(bool)
pred_federado_bool = (prob_gt_federado >= THRESHOLD_FEDERADO).astype(bool)

# Validaciones antes de exportar
assert len(pred_central_bool)  == 100_000, "Guatemala debe tener 100,000 filas"
assert len(pred_federado_bool) == 100_000, "Guatemala debe tener 100,000 filas"

print(f"Fraudes en 30% - Centralizado: {pred_central_bool[:N_30PCT].sum()}")
print(f"Fraudes en 30% - Federado:     {pred_federado_bool[:N_30PCT].sum()}")

Fraudes en 30% - Centralizado: 809
Fraudes en 30% - Federado:     730


In [107]:
import openpyxl

# --- Entrega 1: Modelo centralizado ---
df_30pct_central = pd.DataFrame({
    "is_fraud": pred_central_bool[:N_30PCT]
})
df_30pct_central["is_fraud"] = df_30pct_central["is_fraud"].map({True: "True", False: "False"})

ruta_30pct_central = "resultados_objetivo_b/inferencia_30pct_centralizado.xlsx"
df_30pct_central.to_excel(ruta_30pct_central, index=False)

# --- Entrega 2: Modelo federado ---
df_30pct_federado = pd.DataFrame({
    "is_fraud": pred_federado_bool[:N_30PCT]
})
df_30pct_federado["is_fraud"] = df_30pct_federado["is_fraud"].map({True: "True", False: "False"})

ruta_30pct_federado = "resultados_objetivo_b/inferencia_30pct_federado.xlsx"
df_30pct_federado.to_excel(ruta_30pct_federado, index=False)

print(f"Archivo centralizado: {ruta_30pct_central}")
print(f"  Filas: {len(df_30pct_central)}")
print(f"  Fraudes: {(df_30pct_central['is_fraud'] == 'True').sum()}")

print(f"\nArchivo federado: {ruta_30pct_federado}")
print(f"  Filas: {len(df_30pct_federado)}")
print(f"  Fraudes: {(df_30pct_federado['is_fraud'] == 'True').sum()}")

Archivo centralizado: resultados_objetivo_b/inferencia_30pct_centralizado.xlsx
  Filas: 30000
  Fraudes: 809

Archivo federado: resultados_objetivo_b/inferencia_30pct_federado.xlsx
  Filas: 30000
  Fraudes: 730


### 14.5 Guardar CSV interno con probabilidades completas (análisis interno)

Este archivo NO se envía a PlusTI. Es para análisis interno y para poder
recalibrar el threshold cuando llegue el feedback del 30%.

In [108]:
df_gt_probs = pd.DataFrame({
    "prob_centralizado":  prob_gt_central,
    "prob_bo_individual": prob_gt_bo,
    "prob_br_individual": prob_gt_br,
    "prob_federado":      prob_gt_federado,
    "pred_centralizado":  pred_central_bool,
    "pred_federado":      pred_federado_bool,
})

df_gt_probs.to_csv(
    "resultados_objetivo_b/probabilidades_guatemala_completo.csv",
    index=False
)

print("CSV interno guardado: probabilidades_guatemala_completo.csv")
print(f"  Filas: {len(df_gt_probs)}")
print(df_gt_probs.describe().round(4))

CSV interno guardado: probabilidades_guatemala_completo.csv
  Filas: 100000
       prob_centralizado  prob_bo_individual  prob_br_individual  \
count        100000.0000         100000.0000         100000.0000   
mean              0.0570              0.0334              0.0292   
std               0.1599              0.1393              0.1397   
min               0.0001              0.0000              0.0000   
25%               0.0111              0.0036              0.0019   
50%               0.0219              0.0068              0.0054   
75%               0.0384              0.0152              0.0106   
max               0.9999              0.9999              1.0000   

       prob_federado  
count    100000.0000  
mean          0.0312  
std           0.1258  
min           0.0000  
25%           0.0038  
50%           0.0071  
75%           0.0127  
max           0.9999  
